# Ultimate Data Science Challenge — Walkthrough Notebook

This notebook is written in a **learning-heavy style**: it solves the challenge while also explaining the reasoning behind each step.

We will cover:

1. **Part 1 — Exploratory Data Analysis**
   - Aggregate login timestamps into 15-minute intervals
   - Visualize the time series
   - Describe daily and weekly demand patterns
   - Check for data quality issues

2. **Part 2 — Experiment and Metrics Design**
   - Propose a success metric
   - Design a practical experiment
   - Explain statistical testing and interpretation

3. **Part 3 — Predictive Modeling**
   - Define retention
   - Clean and explore the rider dataset
   - Build predictive models
   - Interpret the results in business terms


## Imports and setup


In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


## File paths

These assume the notebook sits in the same folder as the two JSON files.  
If your files are elsewhere, update the paths below.


In [ ]:
LOGIN_PATH = "logins.json"
RETENTION_PATH = "ultimate_data_challenge.json"


# Part 1 — Exploratory Data Analysis
The challenge asks us to aggregate login counts into **15-minute intervals** and characterize the underlying demand patterns.


### Step 1: Load the login data


In [ ]:
with open(LOGIN_PATH, "r") as f:
    logins_raw = json.load(f)

logins = pd.DataFrame(logins_raw)
logins["login_time"] = pd.to_datetime(logins["login_time"])

print("Shape:", logins.shape)
display(logins.head())
print("
Date range:", logins["login_time"].min(), "to", logins["login_time"].max())


### Step 2: Basic data quality checks

Before plotting anything, it is good practice to check:

- missing values
- duplicate timestamps
- whether the data spans a reasonable time range
- whether timestamps are parseable


In [ ]:
print("Missing values:")
display(logins.isna().sum())

duplicate_rows = logins.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows:,}")


**Interpretation:**  
Duplicate timestamps are not automatically an error here. Multiple users can log in at the exact same second, so duplicate timestamp rows may reflect real simultaneous events. Still, it is worth noting in the analysis as a potential data quality consideration.


### Step 3: Aggregate to 15-minute intervals


In [ ]:
login_15min = (
    logins
    .set_index("login_time")
    .resample("15min")
    .size()
    .rename("login_count")
    .to_frame()
)

login_15min["hour"] = login_15min.index.hour
login_15min["dayofweek"] = login_15min.index.dayofweek
login_15min["weekday_name"] = login_15min.index.day_name()
login_15min["is_weekend"] = login_15min["dayofweek"] >= 5
login_15min["minutes_since_midnight"] = (
    login_15min.index.hour * 60 + login_15min.index.minute
)

display(login_15min.head())
print(login_15min["login_count"].describe())


### Step 4: Plot the full time series


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(login_15min.index, login_15min["login_count"])
plt.title("Login Counts Aggregated into 15-Minute Intervals")
plt.xlabel("Date")
plt.ylabel("Login Count")
plt.tight_layout()
plt.show()


### Step 5: Average hourly demand

The full time series can look noisy. A useful next step is to average demand by hour of day to reveal the daily cycle.


In [ ]:
hourly_profile = login_15min.groupby("hour")["login_count"].mean()

plt.figure(figsize=(10, 4))
plt.plot(hourly_profile.index, hourly_profile.values, marker="o")
plt.title("Average Login Count by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Average 15-Minute Login Count")
plt.xticks(range(24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(hourly_profile.sort_values(ascending=False).head(10).rename("avg_login_count"))


### Step 6: Average demand by day of week


In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_profile = (
    login_15min.groupby("weekday_name")["login_count"]
    .mean()
    .reindex(weekday_order)
)

plt.figure(figsize=(10, 4))
plt.plot(dow_profile.index, dow_profile.values, marker="o")
plt.title("Average Login Count by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average 15-Minute Login Count")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(dow_profile.rename("avg_login_count"))


### Step 7: Weekday vs weekend intraday pattern


In [ ]:
intraday_profile = (
    login_15min
    .groupby(["is_weekend", "minutes_since_midnight"])["login_count"]
    .mean()
    .reset_index()
)

weekday_intraday = intraday_profile[intraday_profile["is_weekend"] == False]
weekend_intraday = intraday_profile[intraday_profile["is_weekend"] == True]

plt.figure(figsize=(12, 5))
plt.plot(weekday_intraday["minutes_since_midnight"] / 60, weekday_intraday["login_count"], label="Weekday")
plt.plot(weekend_intraday["minutes_since_midnight"] / 60, weekend_intraday["login_count"], label="Weekend")
plt.title("Average Intraday Login Pattern: Weekday vs Weekend")
plt.xlabel("Hour of Day")
plt.ylabel("Average 15-Minute Login Count")
plt.xticks(range(0, 25, 2))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Step 8: Peak intervals


In [ ]:
top_intervals = login_15min["login_count"].sort_values(ascending=False).head(10)
display(top_intervals)


### Part 1 summary

From the dataset:

- There are **93,142 login records**.
- The data runs from **1970-01-01 20:12:16** to **1970-04-13 18:57:38**.
- The **maximum** number of logins in a 15-minute interval is **73**, occurring at **1970-03-01 04:30:00**.
- There are **877 exact duplicate rows**, which may simply reflect multiple logins recorded at identical timestamps.

**Business interpretation**
- Demand is **strongest late at night**, especially around **10 PM to 2 AM**.
- Activity is **lowest in the early morning / daytime** relative to the night peak.
- Demand builds through the week and is strongest on **Friday, Saturday, and Sunday**.
- Weekend behavior is meaningfully different from weekday behavior, which suggests the platform is influenced by nightlife / late social activity rather than purely commuting.

**Data quality note**
- I would mention duplicates in the write-up, but I would **not automatically delete them** without proof they are system duplicates rather than legitimate simultaneous user events.


# Part 2 — Experiment and Metrics Design

The business question is whether reimbursing tolls would encourage drivers to serve both cities instead of staying exclusive to one side.

## 1) Key measure of success

My primary metric would be:

### **Cross-city driver participation rate**
> The proportion of active driver partners who complete at least one trip in **both** cities during the experiment window.

### Why this metric?
Because the stated goal is not simply “more trips” or “more revenue.” The stated goal is to encourage drivers to be **available in both cities**. This metric directly measures the behavioral change that management cares about.

## Supporting metrics
I would also track:

- **Cross-city trip share**: percent of trips completed by drivers who serve both cities
- **Rider ETAs / fulfillment rates** in both cities
- **Driver earnings net of tolls**
- **Incremental trips and gross bookings**
- **Cost per additional cross-city driver activated**

These secondary metrics matter because even if cross-city participation rises, the program may still be too expensive or may hurt marketplace balance elsewhere.


## 2) Practical experiment design

### a) How to implement the experiment
A practical design is a **randomized controlled experiment at the driver level**.

- Randomly assign eligible driver partners into:
  - **Treatment**: toll reimbursements enabled
  - **Control**: current system, no reimbursement
- Run the test long enough to capture both **weekday and weekend** behavior, ideally at least **2–4 weeks**
- Use the same reimbursement rules throughout the test
- Track trips, crossings, and city activity for both groups

### Why randomize at the driver level?
Because the intervention changes driver incentives. Driver-level randomization directly measures whether the policy changes driver behavior.

### Important implementation notes
- Stratify randomization by baseline city, tenure, and activity level so the groups are balanced
- Exclude special operational events if possible
- Ensure toll reimbursements are easy to understand, visible, and actually paid quickly


### b) Statistical tests

Because the primary metric is a **proportion** (whether a driver served both cities or not), a natural primary test is:

- **Two-proportion z-test**  
or equivalently
- **Chi-square test of independence**

If I model the outcome with controls, I would also use:

- **Logistic regression**
  - Outcome: served both cities (yes/no)
  - Predictor: treatment indicator
  - Optional controls: baseline activity, city, driver tenure, time-of-week effects

For secondary continuous outcomes such as earnings or trips:
- **t-test** if assumptions are reasonable
- otherwise a **nonparametric alternative** or bootstrap confidence intervals


### c) How to interpret the results

I would compare treatment vs control on the primary metric:

- If treatment significantly increases cross-city driver participation and the economics are favorable, I would recommend expanding the program.
- If participation improves but only at very high reimbursement cost, I would recommend a more targeted version (for example, only during certain hours or only for under-supplied routes).
- If there is no significant improvement, I would recommend not rolling it out broadly.

### Caveats
- Drivers may respond differently on weekdays vs weekends
- There may be spillover effects if treatment drivers change supply conditions for control drivers
- Short-term novelty effects may overstate long-term impact
- The operationally correct answer may be a **targeted reimbursement policy**, not necessarily universal reimbursement


# Part 3 — Predictive Modeling

1. Perform any cleaning, exploratory analysis, and/or visualizations to use the provided
data for this analysis (a few sentences/plots describing your approach will suffice). What
fraction of the observed users were retained?

2. Build a predictive model to help Ultimate determine whether or not a user will be active in
their 6th month on the system. Discuss why you chose your approach, what alternatives
you considered, and any concerns you have. How valid is your model? Include any key
indicators of model performance.

3. Briefly discuss how Ultimate might leverage the insights gained from the model to
improve its long term rider retention (again, a few sentences will suffice).

---

The goal is to predict whether a **rider** is **retained**, where retention is defined as being active in the **preceding 30 days** relative to the data pull date.


### Step 1: Load the rider dataset


In [ ]:
ultimate = pd.read_json(RETENTION_PATH)

print("Shape:", ultimate.shape)
display(ultimate.head())


### Step 2: Parse dates and define retention

The challenge says the data was pulled several months later and a user is retained if they were active in the preceding 30 days.

A common practical approach is:
- use the **latest last_trip_date in the dataset** as the data pull reference point
- label a user as retained if their `last_trip_date` is within 30 days of that reference date


In [ ]:
ultimate["signup_date"] = pd.to_datetime(ultimate["signup_date"])
ultimate["last_trip_date"] = pd.to_datetime(ultimate["last_trip_date"])

observation_date = ultimate["last_trip_date"].max()
ultimate["retained"] = ultimate["last_trip_date"] >= (observation_date - pd.Timedelta(days=30))

print("Observation date:", observation_date)
print("Retention rate:", ultimate["retained"].mean())


### Step 3: Missing value check


In [ ]:
missing_pct = (ultimate.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.to_frame("missing_percent"))


**Interpretation:**  
The largest missingness is in `avg_rating_of_driver`, which is understandable because users with very few trips may never have enough ratings recorded. There is also a small amount of missingness in `phone` and `avg_rating_by_driver`.


### Step 4: A quick look at retention by key categories


In [ ]:
city_retention = ultimate.groupby("city")["retained"].mean().sort_values(ascending=False)
phone_retention = ultimate.groupby("phone")["retained"].mean().sort_values(ascending=False)
black_retention = ultimate.groupby("ultimate_black_user")["retained"].mean()

display(city_retention.rename("retention_rate"))
display(phone_retention.rename("retention_rate"))
display(black_retention.rename("retention_rate"))


A few patterns stand out immediately:

- Overall retention is about **37.61%**
- Users in **King's Landing** retain at a much higher rate than users in the other cities
- **iPhone** users retain more than Android users in this sample
- Users who used **Ultimate Black** in the first 30 days appear more likely to be retained

These are associations, not necessarily causal effects.


### Step 5: Prepare features for modeling

We should avoid using raw `last_trip_date` directly as a feature because it would leak the answer into the model.  
Similarly, `signup_date` is not directly useful in raw form, so we extract a lighter feature: signup day of week.


In [ ]:
ultimate["signup_dayofweek"] = ultimate["signup_date"].dt.day_name()

X = ultimate.drop(columns=["retained", "signup_date", "last_trip_date"])
y = ultimate["retained"].astype(int)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


### Step 6: Train / test split and preprocessing

We will:
- impute missing numeric values with the median
- scale numeric variables for logistic regression
- impute missing categorical values with the most frequent category
- one-hot encode categorical variables


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


### Step 7: Build two models

I am using:
1. **Logistic Regression** as an interpretable baseline
2. **Random Forest** as a stronger nonlinear model that can capture interactions automatically

This is a good interview pattern because it shows both interpretability and performance awareness.


In [ ]:
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

random_forest = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

models = {
    "Logistic Regression": log_reg,
    "Random Forest": random_forest
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs)
    })

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
display(results_df)


### Step 8: Confusion matrix for the best model


In [ ]:
best_model = random_forest
best_model.fit(X_train, y_train)

best_preds = best_model.predict(X_test)
best_probs = best_model.predict_proba(X_test)[:, 1]

ConfusionMatrixDisplay.from_predictions(y_test, best_preds)
plt.title("Random Forest Confusion Matrix")
plt.show()


### Step 9: Feature importance

For the random forest, feature importances give a rough sense of which variables the model used most strongly.  
These are not causal effects, but they are useful for identifying strong predictors.


In [ ]:
ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(categorical_features))
feature_names = numeric_features + cat_feature_names

rf_importances = pd.Series(
    best_model.named_steps["model"].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(rf_importances.head(15).to_frame("importance"))


## Part 3 model discussion

### Why these models?
- **Logistic regression** is easy to explain and gives a strong baseline.
- **Random forest** handles nonlinear relationships and interactions without much manual feature engineering.

### What alternatives could be considered?
- Gradient boosting models such as XGBoost or LightGBM
- Regularized logistic regression
- Calibrated probability models if retention probabilities need to drive CRM interventions

### How valid is the model?
On the held-out test set:

- **Logistic Regression ROC-AUC:** about **0.755**
- **Random Forest ROC-AUC:** about **0.852**

That tells us the random forest is materially better at separating retained from non-retained users in this sample.

### Important predictive signals
From the exploratory breakdowns and feature importance, likely strong predictors include:
- `trips_in_first_30_days`
- `ultimate_black_user`
- `city`
- `phone`
- usage pattern variables such as `weekday_pct`, `surge_pct`, and `avg_surge`

In plain English: riders who engage more in their first month and adopt premium / higher-usage behaviors tend to retain better.


## Business recommendations for Ultimate

Ultimate could use these results in several practical ways:

1. **Identify at-risk users early**
   - Score users near the end of their first month
   - Send targeted incentives or re-engagement offers to those with low predicted retention

2. **Increase first-30-day engagement**
   - `trips_in_first_30_days` is one of the strongest predictors
   - This suggests onboarding and early activation matter a lot

3. **Tailor retention strategy by city**
   - City appears to matter substantially
   - That may reflect differences in marketplace quality, supply, pricing, or product fit

4. **Test premium-feature exposure carefully**
   - Ultimate Black usage is associated with higher retention
   - A smart next step would be an experiment offering limited premium trials to selected users

## Caveats
- This is observational data, so the model identifies predictors, not guaranteed causes
- Retention is defined relative to the available data pull date, so the exact label depends on that snapshot
- More temporal product and trip history features would likely improve performance further


## Final conclusion

This notebook answers all three parts of the challenge:

- **Part 1:** late-night and weekend demand patterns are clearly visible after aggregating logins into 15-minute intervals
- **Part 2:** the most direct success metric is whether more drivers serve both cities, measured with a randomized experiment
- **Part 3:** retention can be predicted reasonably well, with a random forest outperforming logistic regression, and early engagement appearing especially important

If this were a real submission, I would include:
- this notebook
- the accompanying data files locally (but not necessarily committed to GitHub if prohibited)
- a short executive summary in the opening markdown or README
